In [9]:
libs_path = os.path.join(
    os.getcwd().replace('Projects 3.12\\Options. ThetaData', ''), 'Projects'
)

In [11]:
import pandas as pd
import json
import numpy as np
import polars as pl
import plotly.express as px
import time
import datetime
import sys
import os
import matplotlib
import pickle
import requests
import logging

libs_path = os.path.join(
    os.getcwd().replace('Projects 3.12\\Options. ThetaData', ''), 'Projects'
)
sys.path.append(libs_path)
from FinanceAndMl_libs import finance_ml as fm

from tqdm.notebook import tqdm_notebook as tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pprint import pprint
from io import StringIO

from requests.adapters import ConnectionError, ReadTimeout, ReadTimeoutError
from urllib3.connection import NewConnectionError
from urllib3.util.retry import MaxRetryError
from urllib3.exceptions import ConnectTimeoutError

# Settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 100
pd.options.display.width = 20000
pd.set_option('display.float_format', lambda x: '%.4f' % x)
np.set_printoptions(suppress=True)
pl.Config.set_tbl_rows(100)

from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

# Load Tickers

In [2]:
need_tickers = []
for core, folder, files in os.walk('data\stocks\daily'):
    for file in tqdm(files):
        cur_df = pl.read_parquet(os.path.join(core, file))
        if (cur_df['rol_50_vol'] > 1_000_000).any():
            need_tickers.append(file.split('_')[2])

  0%|          | 0/7804 [00:00<?, ?it/s]

In [26]:
data_path = os.path.join(os.getcwd().split('Projects')[0], 'Цены\Дейли')
tickers_paths = {}
for dir, folders, files in os.walk(data_path):
    if (r'tiingo' in dir) or ('yahoo' in dir) or ('tos' in dir) or ('sintetic' in dir) or ('pandas' in dir):
        tickers_paths[dir] = files

tiingo_paths = [path for path in tickers_paths.keys() if 'tiingo' in path]
tiingo_paths = [path for path in tiingo_paths if '\\USA\\' in path]
tiingo_paths.reverse()

total_df = pl.DataFrame()
for ticker in tqdm(need_tickers):
    t_csv = f'{ticker}.csv'

    for tiingo_path in tiingo_paths:
        if t_csv in tickers_paths[tiingo_path]:
            cur_path = os.path.join(tiingo_path, t_csv)
            print(cur_path)
            
            cur_df = pl.read_csv(cur_path).with_columns(pl.lit(ticker).alias('ticker'))\
                [['date', 'close', 'adjClose', 'volume', 'divCash', 'splitFactor', 'ticker']]
                
            if len(cur_df) == 0:
                continue
            total_df = pl.concat([total_df, cur_df])
            break

total_df = total_df\
    .with_columns(pl.col('date').str.slice(0, 10).str.to_date("%Y-%m-%d"))\
    .filter(pl.col('date') >= datetime.date(2015, 1, 1))\
    .sort('ticker', 'date')\
    .with_columns(pl.col('adjClose').pct_change().over('ticker').alias('adjClose_pct'))

  0%|          | 0/4670 [00:00<?, ?it/s]

G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\BDBD.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\PMCS.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\IACI.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\DYAX.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\UTIW.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\MDAS.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\OVTI.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\BRCM.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\OCAT.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\VPCO.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\EZCH.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\TSYS.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\GMCR.csv
G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA\Stock\NASDAQ\RJET.csv
G:\Биржа\Stocks. BigData\Цены\Дейл

# Create features

## SMA & STD

In [32]:
period_l = 200
sma_df = total_df[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period_l}i")\
    .agg(
        pl.col('date').first(), 
        pl.col('adjClose').mean().alias(f'sma{period_l}'),
        pl.col('adjClose').std().alias(f'std{period_l}'),
        pl.col('adjClose_pct').std().alias(f'std%{period_l}')
    ).unique(subset=['ticker', 'date'], keep='last')\
    .sort('ticker', 'date')
features_df = total_df.join(
    sma_df[['ticker', 'date', f'sma{period_l}', f'std{period_l}', f'std%{period_l}']], 
    on=['ticker', 'date'], 
    how='inner'
)
len(features_df) == len(total_df)

True

In [33]:
period_m = 50
sma_df = total_df[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period_m}i")\
    .agg(
        pl.col('date').first(), 
        pl.col('adjClose').mean().alias(f'sma{period_m}'), 
        pl.col('adjClose').std().alias(f'std{period_m}'),
        pl.col('adjClose_pct').std().alias(f'std%{period_m}')
    ).unique(subset=['ticker', 'date'], keep='last')\
    .sort('ticker', 'date')
features_df = features_df.join(
    sma_df[['ticker', 'date', f'sma{period_m}', f'std%{period_m}']], 
    on=['ticker', 'date'], 
    how='inner'
)
len(features_df) == len(total_df)

True

In [34]:
period_s = 20
sma_df = total_df[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period_s}i")\
    .agg(
        pl.col('date').first(), 
        pl.col('adjClose').mean().alias(f'sma{period_s}'), 
        pl.col('adjClose').std().alias(f'std{period_s}'),
        pl.col('adjClose_pct').std().alias(f'std%{period_s}')
    ).unique(subset=['ticker', 'date'], keep='last')\
    .sort('ticker', 'date')
features_df = features_df.join(
    sma_df[['ticker', 'date', f'sma{period_s}', f'std%{period_s}']], 
    on=['ticker', 'date'], 
    how='inner'
)
len(features_df) == len(total_df)

True

In [35]:
features_df = features_df.with_columns(
    (((pl.col(f'sma{period_l}') + pl.col(f'std{period_l}') * 3) / pl.col('adjClose') - 1) * 100).alias(f'+3stdSma{period_l}'),
    (((pl.col(f'sma{period_l}') - pl.col(f'std{period_l}') * 3) / pl.col('adjClose') - 1) * 100).alias(f'-3stdSma{period_l}'),
    
    ((pl.col('adjClose') / pl.col(f'sma{period_l}') - 1) * 100).alias(f'sma{period_l}'),
    ((pl.col('adjClose') / pl.col(f'sma{period_m}') - 1) * 100).alias(f'sma{period_m}'),
    ((pl.col('adjClose') / pl.col(f'sma{period_s}') - 1) * 100).alias(f'sma{period_s}'),

    (pl.col('std%20') / pl.col('std%200')).alias('std%20_std%200'),
    (pl.col('std%50') / pl.col('std%200')).alias('std%50_std%200'),
    (pl.col('std%20') / pl.col('std%50')).alias('std%20_std%50')
).drop('std200')

In [36]:
std_df = total_df.sort('ticker', 'date')\
    .rolling(index_column='date', period='1y', group_by='ticker')\
    .agg(pl.col('adjClose_pct').std().alias('std1Y'))\
    .with_columns((pl.col('std1Y') * np.sqrt(252)).alias('annualStd'))
features_df = features_df.join(std_df[['ticker', 'date', 'annualStd']], on=['ticker', 'date'], how='inner')
len(features_df) == len(total_df)

True

In [37]:
scal_df = features_df\
    .sort('ticker', 'date')\
    .rolling(index_column='date', period='1y', group_by='ticker')\
    .agg(pl.col('std%20').min().alias('std%20_1Ymin'), pl.col('std%20').max().alias('std%20_1Ymax'))

features_df = features_df.join(scal_df, on=['ticker', 'date'], how='inner')\
    .with_columns(
        ((pl.col('std%20') - pl.col('std%20_1Ymin')) / (pl.col('std%20_1Ymax') - pl.col('std%20_1Ymin'))).alias('std%20_1Yscal')
    ).drop(['std%20_1Ymax', 'std%20_1Ymin'])

## ATR

In [38]:
atr_df = features_df.with_columns(
    pl.col('adjClose').shift(5).over('ticker').alias('adjClose_shift5'),
    pl.col('adjClose').shift(20).over('ticker').alias('adjClose_shift20'),
    pl.col('adjClose').shift(100).over('ticker').alias('adjClose_shift100'),
).with_columns(
    ((pl.col('adjClose') / pl.col('adjClose_shift5') - 1)).over('ticker').alias('adjClose_shift5_pct'),
    ((pl.col('adjClose') / pl.col('adjClose_shift20') - 1)).over('ticker').alias('adjClose_shift20_pct'),
    ((pl.col('adjClose') / pl.col('adjClose_shift100') - 1)).over('ticker').alias('adjClose_shift100_pct'),
).sort('ticker', 'date')\
 .rolling(index_column='date', period='1y', group_by='ticker')\
 .agg(
     pl.col('adjClose_shift5_pct').abs().median().alias('adjC_shift5_pct_med'),
     pl.col('adjClose_shift20_pct').abs().median().alias('adjC_shift20_pct_med'),
     pl.col('adjClose_shift100_pct').abs().median().alias('adjC_shift100_pct_med')
 )

features_df = features_df.join(atr_df, on=['ticker', 'date'], how='inner')

## Range H_L

In [39]:
max_low_1y = features_df.sort('ticker', 'date')\
    .rolling(index_column='date', period='1y', group_by='ticker')\
    .agg(
        pl.col('adjClose').max().alias(f'adjC_1Ymax'), 
        pl.col('adjClose').min().alias(f'adjC_1Ymin'),
    )

features_df = features_df.join(max_low_1y, on=['ticker', 'date'], how='inner')

In [40]:
period: int = 20

max_low_Ndays = features_df.sort('ticker', 'date')[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period}i")\
    .agg(
        pl.col('date').first(), 
        pl.col('adjClose').max().alias(f'adjC_{period}max'),
        pl.col('adjClose').min().alias(f'adjC_{period}min'),
    ).unique(subset=['ticker', 'date'], keep='last')\
    .with_columns((pl.col(f'adjC_{period}max') - pl.col(f'adjC_{period}min')).alias(f'MaxMin_{period}diff'))

features_df = features_df.join(
    max_low_Ndays[['ticker', 'date', f'MaxMin_{period}diff']], 
    on=['ticker', 'date'], 
    how='inner'
)

In [41]:
period: int = 100

max_low_Ndays = features_df.sort('ticker', 'date')[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period}i")\
    .agg(
        pl.col('date').first(), 
        pl.col('adjClose').max().alias(f'adjC_{period}max'),
        pl.col('adjClose').min().alias(f'adjC_{period}min'),
    ).unique(subset=['ticker', 'date'], keep='last')\
    .with_columns((pl.col(f'adjC_{period}max') - pl.col(f'adjC_{period}min')).alias(f'MaxMin_{period}diff'))

features_df = features_df.join(
    max_low_Ndays[['ticker', 'date', f'MaxMin_{period}diff']], 
    on=['ticker', 'date'], 
    how='inner'
)

In [42]:
features_df = features_df.with_columns(
    ((pl.col('adjClose') - pl.col('adjC_1Ymin')) / (pl.col('adjC_1Ymax') - pl.col('adjC_1Ymin'))).alias('adjC_1Yrange'),
    (pl.col('MaxMin_20diff') / (pl.col('adjC_1Ymax') - pl.col('adjC_1Ymin'))).alias('MaxMin20_MaxMin1Y'),
    (pl.col('MaxMin_100diff') / (pl.col('adjC_1Ymax') - pl.col('adjC_1Ymin'))).alias('MaxMin100_MaxMin1Y')
).drop('adjC_1Ymax', 'adjC_1Ymin', 'MaxMin_20diff', 'MaxMin_100diff')

## RSI

In [43]:
period: int = 14
period_l: int = 28

rsi_df = total_df.with_columns(
    pl.col('adjClose').diff().alias('price_diff').over('ticker')
).with_columns(
    pl.col('price_diff').ewm_mean(alpha=1.0 / period, adjust=False, ignore_nulls=True).over('ticker').alias(f'cng_avg_{period}'),
    pl.col('price_diff').abs().ewm_mean(alpha=1.0 / period, adjust=False, ignore_nulls=True).over('ticker').alias(f'abs_cng_avg_{period}'),

    pl.col('price_diff').ewm_mean(alpha=1.0 / period_l, adjust=False, ignore_nulls=True).over('ticker').alias(f'cng_avg_{period_l}'),
    pl.col('price_diff').abs().ewm_mean(alpha=1.0 / period_l, adjust=False, ignore_nulls=True).over('ticker').alias(f'abs_cng_avg_{period_l}')
).with_columns(
    pl.when(pl.col(f'abs_cng_avg_{period}') != 0).then(pl.col(f'cng_avg_{period}') / pl.col(f'abs_cng_avg_{period}')).otherwise(0).alias(f'cng_ratio_{period}'),
    pl.when(pl.col(f'abs_cng_avg_{period_l}') != 0).then(pl.col(f'cng_avg_{period_l}') / pl.col(f'abs_cng_avg_{period_l}')).otherwise(0).alias(f'cng_ratio_{period_l}')
).with_columns(
    (50 * (pl.col(f'cng_ratio_{period}') + 1)).alias(f'RSI{period}'),
    (50 * (pl.col(f'cng_ratio_{period_l}') + 1)).alias(f'RSI{period_l}')
).sort('ticker', 'date')

features_df = features_df.join(rsi_df[['ticker', 'date', f'RSI{period}', f'RSI{period_l}']], on=['ticker', 'date'], how='inner')
len(features_df) == len(total_df)

True

## Stochastic

In [44]:
period: int = 14

stochastic_df = features_df.sort('ticker', 'date')[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period}i")\
    .agg(
        pl.col('date').first(),
        pl.col('adjClose').first(),
        pl.col('adjClose').max().alias(f'adjC_{period}max'), 
        pl.col('adjClose').min().alias(f'adjC_{period}min')
    ).unique(subset=['ticker', 'date'], keep='last')\
    .with_columns(
        k_perc=((pl.col('adjClose') - pl.col(f'adjC_{period}min')) / (pl.col(f'adjC_{period}max') - pl.col(f'adjC_{period}min'))) * 100
    ).sort('ticker', 'date')\
    .with_columns(
        pl.col("k_perc").rolling_mean(window_size=3).over('ticker').alias(f'stochast_{period}'),
    )

features_df = features_df.join(stochastic_df[['ticker', 'date', f'stochast_{period}']], on=['ticker', 'date'], how='inner')
len(features_df) == len(total_df)

True

In [45]:
period: int = 21

stochastic_df = features_df.sort('ticker', 'date')[::-1]\
    .with_row_index().with_columns(pl.col('index').cast(pl.Int64))\
    .group_by_dynamic(index_column='index', group_by='ticker', every="1i", period=f"{period}i")\
    .agg(
        pl.col('date').first(),
        pl.col('adjClose').first(),
        pl.col('adjClose').max().alias(f'adjC_{period}max'), 
        pl.col('adjClose').min().alias(f'adjC_{period}min')
    ).unique(subset=['ticker', 'date'], keep='last')\
    .with_columns(
        k_perc=((pl.col('adjClose') - pl.col(f'adjC_{period}min')) / (pl.col(f'adjC_{period}max') - pl.col(f'adjC_{period}min'))) * 100
    ).sort('ticker', 'date')\
    .with_columns(
        pl.col("k_perc").rolling_mean(window_size=21).over('ticker').alias(f'stochast_{period}'),
    )

features_df = features_df.join(stochastic_df[['ticker', 'date', f'stochast_{period}']], on=['ticker', 'date'], how='inner')
len(features_df) == len(total_df)

True

## Volume

In [46]:
period_l = 50
period_s = 5

vol_df = total_df.with_columns(
     (pl.col('volume') * pl.col('close')).alias('dollar_vol')
).with_columns(
    pl.col('volume').rolling_mean(window_size=period_l).over('ticker').alias(f'avg_vol{period_l}'),
    pl.col('volume').rolling_mean(window_size=period_s).over('ticker').alias(f'avg_vol{period_s}'),
    pl.col('dollar_vol').rolling_mean(window_size=period_l).over('ticker').alias(f'dollar_avg_vol{period_l}'),
    pl.col('dollar_vol').rolling_mean(window_size=period_s).over('ticker').alias(f'dollar_avg_vol{period_s}')
).with_columns(
    (pl.col(f'avg_vol{period_s}') / pl.col(f'avg_vol{period_l}')).alias(f'Vol{period_s}toVol{period_l}'),
    (pl.col('volume') / pl.col(f'avg_vol{period_l}')).alias(f'VoltoVol{period_l}'),
    (pl.col(f'dollar_avg_vol{period_s}') / pl.col(f'dollar_avg_vol{period_l}')).alias(f'dollarVol{period_s}toVol{period_l}'),
    (pl.col('dollar_vol') / pl.col(f'dollar_avg_vol{period_l}')).alias(f'dollarVoltoVol{period_l}')
)
features_df = features_df.join(
    vol_df[[
        'ticker', 'date', f'avg_vol{period_l}', f'dollar_avg_vol{period_l}', 
        f'Vol{period_s}toVol{period_l}', f'VoltoVol{period_l}', f'dollarVol{period_s}toVol{period_l}', f'dollarVoltoVol{period_l}'
    ]], 
    on=['ticker', 'date'], 
    how='inner'
)
len(features_df) == len(total_df)

True

## Load SPY and SMA's

In [58]:
path = f'G:/Биржа/Stocks. BigData/Цены/Дейли/tiingo/usa/SPY.csv'
spy_df = pl.read_csv(path, try_parse_dates=True).with_columns(pl.lit('SPY').alias('ticker'))\
    [['date', 'close', 'adjClose', 'volume', 'divCash', 'splitFactor', 'ticker']]

period_l = 200
period_m = 50
period_s = 20
spy_df = spy_df.with_columns(
    pl.col('adjClose').rolling_mean(window_size=period_l).alias(f'sma{period_l}'),
    pl.col('adjClose').rolling_mean(window_size=period_m).alias(f'sma{period_m}'),
    pl.col('adjClose').rolling_mean(window_size=period_s).alias(f'sma{period_s}')
).with_columns(
    ((pl.col('adjClose') / pl.col(f'sma{period_l}') - 1) * 100).alias(f'SPYsma{period_l}'),
    ((pl.col('adjClose') / pl.col(f'sma{period_m}') - 1) * 100).alias(f'SPYsma{period_m}'),
    ((pl.col('adjClose') / pl.col(f'sma{period_s}') - 1) * 100).alias(f'SPYsma{period_s}')
)

In [59]:
features_df = features_df.join(
    spy_df[['date', f'SPYsma{period_l}', f'SPYsma{period_m}', f'SPYsma{period_s}']],
    how='left',
    on='date'
)
len(features_df) == len(total_df)

True

## Divs

In [47]:
div_df = total_df.with_columns(
    (pl.col('divCash') / pl.col('close').shift(1)).alias('divYield')
).rolling(index_column='date', period='1y', group_by='ticker')\
 .agg([pl.sum("divCash").alias("divCashSum"), pl.sum('divYield').alias('divYieldSum')])

features_df = features_df.join(div_df[['ticker', 'date', 'divCashSum', 'divYieldSum']], on=['ticker', 'date'], how='inner')
len(features_df) == len(total_df)

True

# Save

In [48]:
features_df = features_df.with_columns(
    [pl.col(col).cast(pl.Float32) for col in features_df.columns 
     if ('sma' in col) or ('lose' in col) or ('Sma' in col) or ('div' in col) or ('toVol' in col) or \
         ('RSI' in col) or ('split' in col) or ('std' in col) or ('adjC_' in col) or ('MaxMin' in col) or ('stochast' in col)]
)

features_df.write_parquet('data/tiingo_stocks_with_features.parquet', compression='lz4')